# 01 Private and Public

A privacy-sensitive project splits every data and output folder into `private/` and
`public/`. Private folders never reach git; public folders hold only reviewed,
aggregated results. This notebook treats the study's participant records as
identifiable and walks them from a private folder to a public summary.

In [ ]:
import shutil
import subprocess

import framework as fw
import matplotlib.pyplot as plt

root = fw.scaffold()

## The layout

`security.sensitive` is on, and each folder has a private and a public twin.

In [ ]:
print("sensitive:", fw.settings("security.sensitive"))
info = fw.project_info()
info[info["key"].str.contains("private|public")]

## What git ignores

`git check-ignore` shows which rule covers a path. Private folders, `.env`, the cache,
and `framework.db` are ignored; public folders are not.

In [ ]:
paths = [
    "inputs/private/raw/participants.csv",
    "outputs/private/tables/bp_by_participant.csv",
    "outputs/private/cache/model.pickle",
    "inputs/public/final/bp_by_site.csv",
    "outputs/public/tables/bp_by_site.csv",
    ".env",
    "framework.db",
]
for path in paths:
    rule = subprocess.run(["git", "check-ignore", "-v", path], cwd=root,
                          capture_output=True, text=True).stdout.strip()
    print(f"{path:48} {'ignored by ' + rule.split(chr(9))[0] if rule else 'tracked'}")

There is no built-in encryption: protection comes from keeping files in private folders,
out of git, and out of published work.

## Private inputs

Record-level data goes into `inputs/private/raw/`. `data_add()` names each file from
its folder (`inputs.private.raw.participants`), and `data_info()` marks it private.

In [ ]:
shared_csv = root.parent.parent / "shared" / "csv"
private_raw = root / fw.settings("inputs_private_raw")
for name in ("participants", "visits", "sites"):
    shutil.copy(shared_csv / f"{name}.csv", private_raw / f"{name}.csv")
    fw.data_add(private_raw / f"{name}.csv")

fw.data_info("inputs.private.raw.participants")

Work with the records without printing them. Shapes and counts are fine to show.

In [ ]:
participants = fw.data_read("inputs.private.raw.participants")
visits = fw.data_read("inputs.private.raw.visits")
sites = fw.data_read("inputs.private.raw.sites")
records = visits.merge(participants, on="participant_id").merge(sites, on="site_id")
records.shape

The joined records are saved to the private intermediate folder. The catalog name's
leading parts pick the folder.

In [ ]:
fw.data_save(records, "inputs.private.intermediate.bp_records");
fw.data_info("inputs.private.intermediate.bp_records").path

## A public summary

Aggregate to site, arm, and visit, and suppress small cells before anything goes
public. The threshold is 15 participants here, high enough to show suppression on this
small cohort.

In [ ]:
summary = (
    records.dropna(subset=["systolic_bp"])
    .groupby(["site_name", "treatment", "visit_number"])
    .agg(n=("participant_id", "nunique"), mean_systolic=("systolic_bp", "mean"))
    .round(1)
    .reset_index()
)
small = summary["n"] < 15
summary.loc[small, ["n", "mean_systolic"]] = None
print(f"{small.sum()} of {len(summary)} cells suppressed")
fw.data_save(summary, "inputs.public.final.bp_by_site", type="csv");
summary.head()

## Private and public outputs

`save_table()` and the other `save_*()` functions write to the private folder unless
`public=True`. Look at the two paths.

In [ ]:
print(fw.save_table(records.head(50), "bp_records_sample"))
print(fw.save_table(summary, "bp_by_site", public=True))

In [ ]:
means = summary.groupby(["visit_number", "treatment"])["mean_systolic"].mean().unstack()
fig, ax = plt.subplots(figsize=(6, 4))
means.plot(ax=ax, marker="o", xlabel="Visit", ylabel="Mean systolic BP (mmHg)")
fw.save_figure(fig, "bp_by_visit", public=True, width=6, height=4)

`result_list()` records which results are public.

In [ ]:
fw.result_list()[["name", "type", "public"]]

## Publishing

Publishing refuses a file with "private" in its path unless a person confirms it at a
terminal. In a notebook there is no terminal, so it stops. Never pass
`include_private=True` for data you haven't reviewed.

In [ ]:
try:
    fw.publish(root / "outputs/private/tables/bp_records_sample.csv",
               slug="should-not-publish")
except fw.FrameworkError as error:
    print(error)

The public table goes through, to the `local` folder destination (`published/` in the
project root).

In [ ]:
fw.publish(root / "outputs/public/tables/bp_by_site.csv",
           slug="bp-by-site", title="Systolic BP by site (suppressed)")

The check looks at file paths only. A DataFrame in memory has no path, so publishing
`records` directly would not be stopped; publish files from public folders instead.

## Checks

The pre-commit hook is off by default, even here. `data_security` would block commits
of private data; enable it once the project has its own git repository.

In [ ]:
fw.git_hooks_list()

`git_security_audit()` confirms private folders are ignored and no private data is
tracked.

In [ ]:
fw.git_security_audit(history_depth="shallow").passed